# CyberShield LLM Cyber Analyst QLoRA

Bu defter CyberShield için savunma odaklı siber güvenlik asistanı eğitir. Amaç TFLite tespit modellerinin yerini almak değil; olay açıklaması, risk gerekçesi, yanlış alarm kontrolü ve güvenli müdahale önerisi üretmektir.

Varsayılan model 7B sınıfıdır. Colab Pro/L4/A100 önerilir. Ücretsiz GPU yetersiz kalırsa `BASE_MODEL` değerini daha küçük instruct modele düşürün.

In [ ]:
!pip -q install -U "transformers>=4.45" "datasets>=2.20" "accelerate>=0.33" "peft>=0.12" "trl>=0.9" bitsandbytes sentencepiece safetensors


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Bu dosyayı PC'de tools/build_cybershield_llm_dataset.py ile üretip Drive'a koyun.
DATASET_JSONL = '/content/drive/MyDrive/CyberShield/llm/cybershield_sft_dataset.jsonl'
OUTPUT_DIR = '/content/drive/MyDrive/CyberShield/llm/cybershield-cyber-analyst-lora'

# Güçlü öneri: Qwen2.5-7B-Instruct. GPU küçükse 3B/1.5B instruct modele düşürün.
BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'

# İlk eğitimde full 452k yerine dengeli pilot eğitim daha sağlıklıdır.
# Colab T4 için 20k-50k, L4/A100 için 80k-120k önerilir.
MAX_TOTAL_SAMPLES = 100_000
MAX_SAMPLES_PER_DOMAIN = 8_000
MAX_SEQUENCE_LENGTH = 1024

# İlk koşuda eğitimin ilerlediğini kanıtlamak için sınırlı step kullanılır.
# Kalıcı eğitimde None yapabilir veya 2000-5000 gibi artırabilirsiniz.
MAX_TRAIN_STEPS = 1000


In [ ]:
import json
from datasets import load_dataset

dataset = load_dataset('json', data_files=DATASET_JSONL, split='train')

def balanced_subset(ds, max_total=100_000, max_per_domain=8_000):
    counts = {}
    keep = []
    for idx, item in enumerate(ds):
        domain = item.get('metadata', {}).get('domain', 'unknown')
        if counts.get(domain, 0) >= max_per_domain:
            continue
        counts[domain] = counts.get(domain, 0) + 1
        keep.append(idx)
        if len(keep) >= max_total:
            break
    print('Balanced domain counts:', counts)
    return ds.select(keep)

dataset = dataset.shuffle(seed=53)
dataset = balanced_subset(dataset, MAX_TOTAL_SAMPLES, MAX_SAMPLES_PER_DOMAIN)
split = dataset.train_test_split(test_size=0.03, seed=53)
train_ds = split['train']
eval_ds = split['test']
len(train_ds), len(eval_ds), train_ds[0]


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False


In [ ]:
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

def format_chat(example):
    return tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=30,
    lr_scheduler_type='cosine',
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=100,
    save_steps=250,
    save_total_limit=3,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=False,
    max_length=MAX_SEQUENCE_LENGTH,
    packing=False,
    dataset_text_field='text',
    report_to='none',
    max_steps=MAX_TRAIN_STEPS,
)

train_text = train_ds.map(lambda x: {'text': format_chat(x)}, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(lambda x: {'text': format_chat(x)}, remove_columns=eval_ds.column_names)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_text,
    eval_dataset=eval_text,
    peft_config=peft_config,
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


In [ ]:
from transformers import pipeline

def ask(event):
    messages = [
        {'role': 'system', 'content': 'Sen CyberShield içinde çalışan savunma odaklı siber güvenlik analistisin. Sadece güvenli tespit, açıklama ve kullanıcı onaylı müdahale öner.'},
        {'role': 'user', 'content': event},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=420, temperature=0.2, do_sample=True)
    print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

ask('CyberShield olayı: APK indirildi, bilinmeyen kaynak, yüksek izin yoğunluğu, dex string içinde suspicious url var. Kullanıcıya ne söylemeliyim ve hangi müdahale güvenli?')
ask('CyberShield olayı: DNS leak testinde seçili resolver Cloudflare dışında ISP DNS de göründü. Risk ve müdahale ne olmalı?')
ask('CyberShield olayı: WhatsApp görüntülü görüşme sırasında yüksek TLS trafik hacmi var ama domain allowlist içinde. Alarm üretmeli miyim?')


## Yayınlama notu

Bu çıktı Android içinde doğrudan TFLite gibi çalıştırılacak ana tespit modeli değildir. En doğru kullanım: sunucu/API veya cihazın gücüne uygun küçük LLM runtime üzerinden açıklama ve karar desteği. TFLite modeller tespit ve risk skoru üretmeye devam eder; bu LLM olayın nedenini ve güvenli müdahale gerekçesini açıklar.